In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
!rm -rf Cap3_MedicionPobreza
!git clone https://github.com/LizamaD/Cap3_MedicionPobreza.git

Cloning into 'Cap3_MedicionPobreza'...
remote: Enumerating objects: 177, done.
remote: Counting objects: 100% (177/177), done.
remote: Compressing objects: 100% (91/91), done.
remote: Total 177 (delta 99), reused 143 (delta 65), pack-reused 0 (from 0)
Receiving objects: 100% (177/177), 4.39 MiB | 56.25 MiB/s, done.
Resolving deltas: 100% (99/99), done.


In [3]:
import pandas as pd

path_bases = "/content/drive/MyDrive/Doctorado_DavidLizama/datos_tesis/coneval/2024/Bases de datos/"
path_pobreza = '/content/drive/MyDrive/Doctorado_DavidLizama/datos_tesis/coneval/2024/Base final/pobreza_24.csv'


# --- Carga tus datos crudos ---
poblacion_raw = pd.read_csv(f"{path_bases}poblacion.csv")
viviendas_raw = pd.read_csv(f"{path_bases}viviendas.csv")
hogares_raw = pd.read_csv(f"{path_bases}hogares.csv")
trabajos_raw = pd.read_csv(f"{path_bases}trabajos.csv")
gastospersona_raw = pd.read_csv(f"{path_bases}gastospersona.csv")
gastoshogar_raw = pd.read_csv(f"{path_bases}gastoshogar.csv")
ingresos_raw = pd.read_csv(f"{path_bases}ingresos.csv")
pobreza_raw = pd.read_csv(path_pobreza)

/tmp/ipykernel_2778/1323284465.py:8: DtypeWarning: Columns (45) have mixed types. Specify dtype option on import or set low_memory=False.
  poblacion_raw = pd.read_csv(f"{path_bases}poblacion.csv")
/tmp/ipykernel_2778/1323284465.py:9: DtypeWarning: Columns (1,4,27,44) have mixed types. Specify dtype option on import or set low_memory=False.
  viviendas_raw = pd.read_csv(f"{path_bases}viviendas.csv")
/tmp/ipykernel_2778/1323284465.py:10: DtypeWarning: Columns (45,49,53,61,63,69,75,77,81,83,85) have mixed types. Specify dtype option on import or set low_memory=False.
  hogares_raw = pd.read_csv(f"{path_bases}hogares.csv")


In [4]:
%cd /content/Cap3_MedicionPobreza

/content/Cap3_MedicionPobreza


In [5]:
from src.trabajos import process_trabajos
from src.ingresos import generar_ingreso_deflactado_ago2024
from src.gastoshogar import procesar_gastos_enigh
from src.gastospersona import procesar_gastos_persona_enigh
from src.hogares import process_hogares
from src.poblacion import process_poblacion
from src.viviendas import process_viviendas
from src.pipeline import create_master_table, impute_data

In [6]:
# --- Crea la tabla maestra ---
# Esta función procesa, une todo y guarda el resultado en "master_table.csv"
master_df = create_master_table(
    pob_keys=pobreza_raw,
    pob_df=poblacion_raw,
    viv_df=viviendas_raw,
    hog_df=hogares_raw,
    trab_df=trabajos_raw,
    gasper_df=gastospersona_raw,
    gashog_df=gastoshogar_raw,
    ing_df=ingresos_raw,
    output_path=f"{path_bases}enigh_master_table.csv" # Ruta sugerida
)

# --- Imputa los datos faltantes ---
# Esta función toma la tabla maestra y la deja lista para el modelado
df_final_limpio = impute_data(master_df)

# --- Verifica el resultado ---
print("\nDataFrame final después de la imputación:")
print(df_final_limpio.head())

# Confirma que no hay valores nulos
print("\nSuma de nulos por columna en el DF final:")
print(df_final_limpio.isnull().sum().sum()) # Debería ser 0


Procesando tablas individuales...


/content/Cap3_MedicionPobreza/src/ingresos.py:34: FutureWarning: The provided callable <function sum at 0x794aabf28400> is currently using DataFrameGroupBy.sum. In a future version of pandas, the provided callable will be used directly. To keep current behavior pass the string "sum" instead.
  aguinaldo = pd.pivot_table(


Uniendo tablas en una tabla maestra...
Tabla maestra creada con 308444 filas y 160 columnas.
Guardando tabla maestra en '/content/drive/MyDrive/Doctorado_DavidLizama/datos_tesis/coneval/2024/Bases de datos/enigh_master_table.csv'...
Iniciando proceso de imputación de datos...
  - Categórica 'tam_emp': Imputando NaNs con '__MISSING__'.
  - Categórica 'tipoact': Imputando NaNs con '__MISSING__'.
  - Categórica 'socios': Imputando NaNs con '__MISSING__'.
  - Categórica 'soc_nr1': Imputando NaNs con '__MISSING__'.
  - Categórica 'soc_nr2': Imputando NaNs con '__MISSING__'.
  - Categórica 'soc_resp': Imputando NaNs con '__MISSING__'.
  - Categórica 'otra_act': Imputando NaNs con '__MISSING__'.
  - Categórica 'tipoact2': Imputando NaNs con '__MISSING__'.
  - Categórica 'tipoact3': Imputando NaNs con '__MISSING__'.
  - Categórica 'tipoact4': Imputando NaNs con '__MISSING__'.
  - Categórica 'lugar': Imputando NaNs con '__MISSING__'.
  - Categórica 'conf_pers': Imputando NaNs con '__MISSING__'.

/content/Cap3_MedicionPobreza/src/pipeline.py:112: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df_clean[flag_col_name] = df_clean[col].isnull().astype(int)
/content/Cap3_MedicionPobreza/src/pipeline.py:112: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df_clean[flag_col_name] = df_clean[col].isnull().astype(int)
/content/Cap3_MedicionPobreza/src/pipeline.py:112: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all

0


In [ ]:
# Define tu ruta de salida en Google Drive
output_directory = "/content/drive/MyDrive/Doctorado_DavidLizama/datos_tesis/coneval/2024/Bases de datos/"

# Llama a la nueva función para procesar y guardar todo
df_processed, df_pobres, df_no_pobres = prepare_and_split_for_autoencoder(
    df_final_limpio, 
    output_directory
)

# Ahora ya tienes tus DataFrames listos para la estrategia de descubrimiento:
# - df_no_pobres: Para entrenar y validar tu autoencoder.
# - df_pobres: Para pasarlo por el modelo ya entrenado y analizar el error de reconstrucción.


In [7]:
master_df.head(2)

,folioviv,foliohog,numren,pobreza,pobreza_e,parentesco,sexo,edad,afrod,hablaind,...,conf_pers,gasto_per_tri,gas_per_nm_tri,gasto_educ_total,gasto_salud_ind,inscrip,colegia,inst,gasto_persona_total,tiene_gasto_educ
0,100001901,1,1,0.0,0.0,101,1,32,0,0,...,NaN,0.0,20571.4,0.0,0.0,0.0,0.0,00,20571.4,0.0
1,100001901,1,2,0.0,0.0,201,2,24,0,0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [8]:
df_final_limpio.head(2)

,folioviv,foliohog,numren,pobreza,pobreza_e,parentesco,sexo,edad,afrod,hablaind,...,no_ing_imputed,tiene_suel_imputed,gasto_per_tri_imputed,gas_per_nm_tri_imputed,gasto_educ_total_imputed,gasto_salud_ind_imputed,inscrip_imputed,colegia_imputed,gasto_persona_total_imputed,tiene_gasto_educ_imputed
0,100001901,1,1,0.0,0.0,101,1,32,0,0,...,0,0,0,0,0,0,0,0,0,0
1,100001901,1,2,0.0,0.0,201,2,24,0,0,...,0,0,1,1,1,1,1,1,1,1


In [17]:
df_final_limpio.pobreza.value_counts()

,count
pobreza,
0.0,218427
1.0,90017


In [16]:
for x in ['parentesco', 'tipo_viv', 'tenencia', 'tam_emp', 'tipoact', 'socios',
       'soc_nr1', 'soc_nr2', 'soc_resp', 'otra_act', 'tipoact2', 'tipoact3',
       'tipoact4', 'lugar', 'conf_pers', 'inst']:
    print(x)
    print(df_final_limpio[x].value_counts())
    print()

parentesco
parentesco
301    114892
101     91405
201     56705
609     23778
617      6211
603      3434
613      2769
601      2583
303      1607
615      1135
501      1114
618       954
610       919
614       253
612       227
606       140
202        85
205        77
619        32
623        32
602        30
302        20
621        20
616         4
622         4
611         4
605         3
607         2
620         2
608         1
604         1
304         1
Name: count, dtype: int64

tipo_viv
tipo_viv
1      250283
2       43126
4        6620
3        3384
5        2516
7        2214
nan       239
6          62
Name: count, dtype: int64

tenencia
tenencia
4    203372
1     36209
2     33535
3     27882
5      6166
6      1280
Name: count, dtype: int64

tam_emp
tam_emp
__MISSING__    158076
2               54145
1               27727
3               16359
4                7525
11               7379
8                6497
7                6026
5                5869
9              

In [11]:
import numpy as np

# Seleccionar solo las columnas que NO son numéricas
non_numeric_cols = df_final_limpio.select_dtypes(exclude=np.number)

# Ver los nombres de esas columnas
print("Columnas no numéricas:")
print(non_numeric_cols.columns)

# Puedes incluso ver las primeras filas de solo esas columnas
print("\nPrimeras 5 filas de las columnas no numéricas:")
print(non_numeric_cols.head())

Columnas no numéricas:
Index(['parentesco', 'tipo_viv', 'tenencia', 'tam_emp', 'tipoact', 'socios',
       'soc_nr1', 'soc_nr2', 'soc_resp', 'otra_act', 'tipoact2', 'tipoact3',
       'tipoact4', 'lugar', 'conf_pers', 'inst'],
      dtype='object')

Primeras 5 filas de las columnas no numéricas:
  parentesco tipo_viv tenencia      tam_emp      tipoact       socios  \
0        101        7        1           12            0  __MISSING__   
1        201        7        1            2            0  __MISSING__   
2        301        7        1  __MISSING__  __MISSING__  __MISSING__   
3        301        7        1  __MISSING__  __MISSING__  __MISSING__   
4        101        1        3           11            0  __MISSING__   

       soc_nr1      soc_nr2     soc_resp     otra_act     tipoact2  \
0  __MISSING__  __MISSING__  __MISSING__  __MISSING__  __MISSING__   
1  __MISSING__  __MISSING__  __MISSING__  __MISSING__  __MISSING__   
2  __MISSING__  __MISSING__  __MISSING__  __MISSING__ 